# data, ROI, profile API validation

이 notebook은 `src.api`의 단일 공개 API를 셀 단위로 검증한다. 첫 셀의 로컬 경로와 설정을 지정한 뒤 위에서 아래 순서로 실행한다.

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
if not os.path.isfile(os.path.join(PROJECT_ROOT, 'src', 'api.py')):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_PATH = r'REPLACE_WITH_LOCAL_MIM_PATH'
OUTPUT_ROOT = r'REPLACE_WITH_LOCAL_OUTPUT_PATH'
REPORT_ROOT = OUTPUT_ROOT
ROTATION = 0
IMAGE_WIDTH_MM = 100.0
IMAGE_HEIGHT_MM = 50.0
AVERAGE_FILTER_SIZE = 3
REFERENCE_FILTER_SIZE = 101
REFERENCE_PROFILE_CSV = None

if DATA_PATH == r'REPLACE_WITH_LOCAL_MIM_PATH':
    from src.api import create_data

    SYNTHETIC_DATA = create_data(
        os.path.join(PROJECT_ROOT, 'tmp', 'synthetic-validation'),
        num_data=1,
        seed=42,
    )
    DATA_PATH = SYNTHETIC_DATA['data_paths'][0]
else:
    SYNTHETIC_DATA = None

DATA_ROOT = os.path.dirname(DATA_PATH)
BATCH_NAME = os.path.basename(DATA_ROOT)

assert os.path.isfile(DATA_PATH), 'Set DATA_PATH to a readable local .mim file.'

In [ ]:
from src.api import load_data, load_roi, load_profile, show_data, show_roi, show_profile

ROI = {
    'key': 'roi_1',
    'name': 'ROI 1',
    'color': '#ff0000',
    'xmin': 0.1,
    'xmax': 0.4,
    'ymin': 0.2,
    'ymax': 0.8,
}
PROFILE_OPTIONS = {
    'image_width_mm': IMAGE_WIDTH_MM,
    'image_height_mm': IMAGE_HEIGHT_MM,
    'average_filter_size': AVERAGE_FILTER_SIZE,
    'reference_filter_size': REFERENCE_FILTER_SIZE,
}

In [ ]:
image_data = load_data(DATA_PATH, ROTATION)
assert image_data['image'].ndim == 2
assert image_data['source_filename'] == os.path.basename(DATA_PATH)
assert image_data['sample_id'] == os.path.basename(DATA_PATH)[:16]
image_data

In [ ]:
data_figure = show_data(DATA_PATH, ROTATION)
data_figure

In [ ]:
roi_data = load_roi(DATA_PATH, ROTATION, ROI)
x0, y0, width, height = roi_data['pixel_bounds']
assert width > 0 and height > 0
assert roi_data['crop'].shape == (height, width)
roi_data

In [ ]:
roi_figure = show_roi(DATA_PATH, ROTATION, ROI)
roi_figure

In [ ]:
horizontal_profile = load_profile(DATA_PATH, ROTATION, ROI, 'horizontal', **PROFILE_OPTIONS)
assert horizontal_profile['position_axis'] == 'y'
assert horizontal_profile['position_pixel'].size == horizontal_profile['profile_percent'].size
assert horizontal_profile['pixel_to_mm'] > 0
assert horizontal_profile['profile_percent'].dtype.kind == 'f'
horizontal_profile

In [ ]:
vertical_profile = load_profile(DATA_PATH, ROTATION, ROI, 'vertical', **PROFILE_OPTIONS)
assert vertical_profile['position_axis'] == 'x'
assert vertical_profile['position_pixel'].size == vertical_profile['profile_percent'].size
assert vertical_profile['pixel_to_mm'] > 0
assert vertical_profile['profile_percent'].dtype.kind == 'f'
vertical_profile

In [ ]:
horizontal_figure = show_profile(DATA_PATH, ROTATION, ROI, 'horizontal', **PROFILE_OPTIONS)
vertical_figure = show_profile(DATA_PATH, ROTATION, ROI, 'vertical', **PROFILE_OPTIONS)
horizontal_figure, vertical_figure

In [ ]:
try:
    load_data(DATA_PATH, 45)
except ValueError as error:
    assert 'rotation' in str(error).lower()
else:
    raise AssertionError('Unsupported rotation must fail.')

try:
    load_roi(DATA_PATH, ROTATION, {'key': 'bad', 'name': 'Bad', 'color': '#000000', 'xmin': 0.8, 'xmax': 0.2, 'ymin': 0.1, 'ymax': 0.2})
except ValueError as error:
    assert 'ROI' in str(error)
else:
    raise AssertionError('Invalid ROI must fail.')

try:
    load_profile(DATA_PATH, ROTATION, ROI, 'diagonal', **PROFILE_OPTIONS)
except ValueError as error:
    assert 'direction' in str(error).lower()
else:
    raise AssertionError('Unsupported direction must fail.')

try:
    load_profile(DATA_PATH, ROTATION, ROI, 'horizontal', **{**PROFILE_OPTIONS, 'average_filter_size': 0})
except ValueError as error:
    assert 'filter' in str(error).lower()
else:
    raise AssertionError('Invalid filter must fail.')

try:
    load_data(os.path.join(os.path.dirname(DATA_PATH), 'missing-file.mim'), ROTATION)
except FileNotFoundError as error:
    assert 'does not exist' in str(error)
else:
    raise AssertionError('Missing path must fail.')

print('Error validation passed.')

In [ ]:
if REFERENCE_PROFILE_CSV is not None:
    import pandas as pd

    reference_table = pd.read_csv(REFERENCE_PROFILE_CSV, encoding='utf-8')
    current_table = reference_table[reference_table['direction'] == 'horizontal']
    assert len(current_table) == horizontal_profile['profile_percent'].size
    print('Reference path is available. Set project-specific tolerance before numerical comparison.')
else:
    print('Golden comparison skipped. Set REFERENCE_PROFILE_CSV to enable it.')